# Teacher Fine-Tuning: German Customer Support (QLoRA)

Fine-tunes `Qwen2.5-3B-Instruct` on a synthetic German customer support dataset using **QLoRA**.<br>
The resulting LoRA adapter is the **teacher model** used for knowledge distillation in Notebook 3.<br>

**Pipeline position:** `[01 Fine-Tuning]` → 02 Data Generation → 03 Student Distillation → 04 Evaluation

**Strong GPU required.** This notebook was developed on a Kaggle T4 (16 GB VRAM).<br>
[![Open Notebook in Kaggle](https://img.shields.io/badge/Open%20Notebook%20in-Kaggle-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white)](https://www.kaggle.com/code/dennisfeyerabend/01-teacher-finetuning)

## 1. Setup

Install dependencies and check requirements

Local users: skip this cell — install via `pip install -r requirements.txt` instead

In [1]:
%%capture
!pip install -q --upgrade unsloth trl datasets transformers peft bitsandbytes accelerate huggingface_hub

In [2]:
!nvidia-smi
import torch
print(f"PyTorch Version:    {torch.__version__}")
print(f"CUDA Available:     {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Fri May  8 21:35:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Load Base Model

**Why `Qwen2.5-3B-Instruct`?**
- Natively multilingual — understands German without translation
- 3B parameters fit on a T4 GPU with 4-bit quantization
- Instruction-tuned base makes fine-tuning on Q&A data efficient
- Apache 2.0 license

In [3]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
DTYPE = None            # auto-detect: float16 on T4, bfloat16 on Ampere+
LOAD_IN_4BIT = True     # QLoRA: quantize base weights to 4-bit

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)
print(f"Loaded: {model.config._name_or_path}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-3B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Loaded: unsloth/Qwen2.5-3B-Instruct-bnb-4bit


## 3. Apply QLoRA Adapter

LoRA adds small trainable matrices to frozen model layers — only these adapters are updated,
leaving the base weights unchanged. QLoRA combines this with 4-bit quantization,
cutting VRAM usage by ~75% with minimal quality loss.

- `r=16` — adapter rank; controls capacity vs. VRAM trade-off
- `target_modules` — adapters attached to all attention and MLP projections
- `use_gradient_checkpointing="unsloth"` — reduces VRAM by a further ~30%

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    max_seq_length = MAX_SEQ_LENGTH,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

Unsloth 2026.5.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Trainable parameters: 29,933,568 / 1,728,606,208 (1.73%)


## 4. Prepare Dataset

Synthetic German customer support Q&A pairs covering: account issues, payments, shipping, returns, account security, technical problems, newsletter, reviews, and referrals.<br>
All data is synthetic — no real customer data used.<br>

In [5]:
training_data = [
    # --- Account ---
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Ich habe mein Passwort vergessen. Was kann ich tun?",
        "output": "Guten Tag! Kein Problem, das passiert haeufig. Sie koennen Ihr Passwort ganz einfach zuruecksetzen:\n\n1. Klicken Sie auf der Login-Seite auf 'Passwort vergessen'\n2. Geben Sie Ihre E-Mail-Adresse ein\n3. Sie erhalten einen Link zum Zuruecksetzen\n\nFalls Sie weitere Hilfe benoetigen, stehe ich Ihnen gerne zur Verfuegung."
    },
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Wann wird meine Bestellung geliefert?",
        "output": "Vielen Dank fuer Ihre Anfrage! Die Lieferzeit haengt von verschiedenen Faktoren ab:\n\n- Standardversand: 3-5 Werktage\n- Expressversand: 1-2 Werktage\n\nSie koennen den aktuellen Status Ihrer Bestellung jederzeit in Ihrem Kundenkonto unter 'Meine Bestellungen' einsehen. Dort finden Sie auch die Sendungsverfolgungsnummer."
    },
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Ich moechte meine Bestellung stornieren.",
        "output": "Ich verstehe, dass Sie Ihre Bestellung stornieren moechten. Gerne helfe ich Ihnen dabei!\n\nWenn Ihre Bestellung noch nicht versandt wurde, koennen Sie diese kostenlos stornieren:\n1. Gehen Sie zu 'Meine Bestellungen'\n2. Waehlen Sie die betreffende Bestellung\n3. Klicken Sie auf 'Stornieren'\n\nSollte die Bestellung bereits versandt sein, koennen Sie die Annahme verweigern oder die Ware kostenlos zuruecksenden."
    },
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Das Produkt ist defekt angekommen. Was nun?",
        "output": "Das tut mir sehr leid zu hoeren! Ein defektes Produkt ist natuerlich nicht akzeptabel. Wir kuemmern uns sofort darum:\n\n1. Bitte machen Sie Fotos vom Schaden\n2. Kontaktieren Sie uns mit Ihrer Bestellnummer\n3. Wir senden Ihnen umgehend kostenlosen Ersatz oder erstatten den Kaufpreis\n\nSie muessen das defekte Produkt nicht zuruecksenden. Nochmals Entschuldigung fuer die Unannehmlichkeiten!"
    },
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Kann ich den Artikel auch in einer anderen Farbe bekommen?",
        "output": "Selbstverstaendlich! Unsere Artikel sind in verschiedenen Farben erhaeltlich:\n\nSo finden Sie alle Farboptionen:\n1. Oeffnen Sie die Produktseite\n2. Unter dem Produktbild sehen Sie alle verfuegbaren Farben\n3. Klicken Sie auf Ihre Wunschfarbe\n\nFalls Ihre gewuenschte Farbe nicht angezeigt wird, ist sie leider aktuell nicht verfuegbar. Gerne setzen wir Sie auf die Warteliste!"
    },
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Wie kann ich mein Abo kuendigen?",
        "output": "Ich helfe Ihnen gerne bei der Kuendigung Ihres Abonnements. So gehen Sie vor:\n\n1. Loggen Sie sich in Ihr Konto ein\n2. Gehen Sie zu 'Einstellungen' > 'Abonnement'\n3. Klicken Sie auf 'Abo kuendigen'\n4. Bestaetigen Sie die Kuendigung\n\nIhre Kuendigung wird zum Ende des aktuellen Abrechnungszeitraums wirksam. Bis dahin koennen Sie alle Funktionen weiterhin nutzen."
    },
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Gibt es einen Rabattcode?",
        "output": "Vielen Dank fuer Ihr Interesse! Hier sind unsere aktuellen Rabattmoeglichkeiten:\n\n- Newsletter-Anmeldung: 10% Willkommensrabatt\n- Geburtstag: 15% Geburtstagsrabatt (automatisch)\n- Treue-Programm: Ab 5 Bestellungen erhalten Sie Sonderangebote\n\nAktuelle Aktionen finden Sie immer auf unserer Startseite. Darf ich Sie fuer unseren Newsletter anmelden?"
    },
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Die App stuerzt staendig ab.",
        "output": "Das ist aergerlich! Hier sind die Schritte zur Problemloesung:\n\n**Sofortmassnahmen:**\n1. App komplett schliessen und neu starten\n2. Handy neustarten\n3. App-Cache leeren (Einstellungen > Apps > [App] > Cache leeren)\n\n**Falls das Problem besteht:**\n4. App deinstallieren und neu installieren\n5. Pruefen Sie, ob ein Update verfuegbar ist\n\nWenn es weiterhin Probleme gibt, senden Sie uns bitte einen Screenshot der Fehlermeldung. Wir helfen sofort weiter!"
    },
    # --- Zahlungsprobleme ---
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Meine Kreditkarte wurde abgelehnt, obwohl sie gueltig ist.",
        "output": "Das tut mir leid! Eine abgelehnte Karte kann verschiedene Gruende haben:\n\n1. Pruefen Sie, ob Ihre Karte fuer Online-Zahlungen freigeschaltet ist\n2. Kontaktieren Sie Ihre Bank — oft wird die Zahlung als verdaechtig markiert\n3. Versuchen Sie eine alternative Zahlungsmethode (PayPal, Lastschrift)\n\nSollte das Problem bestehen bleiben, helfe ich Ihnen gerne eine alternative Zahlungsart einzurichten!"
    },
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Ich wurde doppelt abgebucht. Was kann ich tun?",
        "output": "Eine Doppelabbuchung ist natuerlich nicht akzeptabel — wir klaeren das sofort!\n\nBitte senden Sie uns:\n1. Ihre Bestellnummer\n2. Einen Screenshot oder Kontoauszug der beiden Abbuchungen\n\nWir pruefen den Vorgang umgehend und erstatten den doppelt abgebuchten Betrag innerhalb von 3-5 Werktagen zurueck. Entschuldigung fuer die Unannehmlichkeiten!"
    },
    # --- Garantie ---
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Wie lange ist die Garantie auf mein Produkt?",
        "output": "Gerne beantworte ich Ihre Frage zur Garantie!\n\nUnsere Garantiezeiten im Ueberblick:\n- Elektronik: 2 Jahre gesetzliche Gewaehrleistung\n- Haushaltsgeraete: 2 Jahre, optional 5 Jahre mit Garantieverlaengerung\n- Kleidung und Accessoires: 1 Jahr\n\nDie genaue Garantiedauer finden Sie auch auf Ihrer Rechnung. Haben Sie ein konkretes Produkt, fuer das ich nachschauen soll?"
    },
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Mein Produkt ist nach 14 Monaten kaputt gegangen. Ist das noch Garantie?",
        "output": "Das ist eine gute Frage! Nach deutschen Gesetzen gilt:\n\n- Innerhalb der ersten 12 Monate: Wir muessen beweisen, dass das Produkt in Ordnung war\n- Nach 12 Monaten: Sie muessen nachweisen, dass es ein Herstellerfehler ist\n\nBitte senden Sie uns Fotos und eine kurze Beschreibung des Problems. Wir pruefen, ob ein Herstellerfehler vorliegt und melden uns innerhalb von 2 Werktagen mit einer Loesung!"
    },
    # --- Account-Sicherheit ---
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Ich glaube, jemand hat sich in mein Konto eingeloggt.",
        "output": "Bitte handeln Sie sofort — Kontosicherheit hat hoechste Prioritaet!\n\n**Sofortmassnahmen:**\n1. Aendern Sie Ihr Passwort jetzt unter 'Einstellungen' > 'Sicherheit'\n2. Aktivieren Sie die Zwei-Faktor-Authentifizierung\n3. Pruefen Sie Ihre letzten Bestellungen auf unbekannte Kaeufe\n\nBitte kontaktieren Sie uns danach mit Ihrer E-Mail-Adresse — wir pruefen die Login-Historie Ihres Kontos und sperren verdaechtige Sitzungen sofort."
    },
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Ich habe eine verdaechtige E-Mail bekommen, die angeblich von euch ist.",
        "output": "Danke, dass Sie uns das melden — das nehmen wir sehr ernst!\n\nWoran Sie echte E-Mails von uns erkennen:\n- Unsere E-Mails kommen immer von @unsereshop.de\n- Wir fragen NIE per E-Mail nach Ihrem Passwort oder Bankdaten\n- Links fuehren immer zu unsereshop.de\n\nBitte klicken Sie keine Links in der verdaechtigen E-Mail an und leiten Sie sie an sicherheit@unsereshop.de weiter. Wir pruefen das sofort!"
    },
    # --- Rueckgabe ---
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Wie kann ich ein Produkt zurueckgeben?",
        "output": "Kein Problem — Ruecksendungen sind bei uns ganz einfach!\n\nSo gehen Sie vor:\n1. Loggen Sie sich ein und gehen Sie zu 'Meine Bestellungen'\n2. Waehlen Sie das Produkt und klicken Sie auf 'Ruecksendung beantragen'\n3. Drucken Sie das kostenlose Ruecksendeetikett aus\n4. Geben Sie das Paket bei einem Paketshop ab\n\nNach Eingang der Ware erstatten wir den Betrag innerhalb von 5 Werktagen. Das Rueckgaberecht gilt 30 Tage ab Lieferdatum."
    },
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Ich habe das falsche Produkt zurueckgeschickt. Was passiert jetzt?",
        "output": "Keine Panik — das koennen wir klaeren!\n\nBitte kontaktieren Sie uns so schnell wie moeglich mit:\n1. Ihrer Bestellnummer\n2. Einer Beschreibung des falsch zurueckgeschickten Artikels\n\nSolange das Paket noch nicht bearbeitet wurde, koennen wir es fuer Sie zurueckhalten. Andernfalls organisieren wir die Ruecksendung an Sie. Bitte handeln Sie schnell, damit wir helfen koennen!"
    },
    # --- Technische Probleme ---
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Die Website laedt bei mir nicht richtig.",
        "output": "Das tut mir leid! Hier sind ein paar schnelle Loesungen:\n\n1. Leeren Sie Ihren Browser-Cache (Strg + Shift + Delete)\n2. Versuchen Sie einen anderen Browser (Chrome, Firefox, Edge)\n3. Deaktivieren Sie Browser-Erweiterungen voruebergehend\n4. Pruefen Sie Ihre Internetverbindung\n\nFalls das Problem weiterhin besteht, teilen Sie uns bitte mit, welchen Browser und welches Betriebssystem Sie verwenden — dann koennen wir gezielter helfen!"
    },
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Ich kann mich nicht in mein Konto einloggen, obwohl mein Passwort korrekt ist.",
        "output": "Das ist frustrierend — lassen Sie uns das gemeinsam loesen!\n\nBitte versuchen Sie folgendes:\n1. Pruefen Sie, ob die Feststelltaste aktiv ist\n2. Loeschen Sie Cookies und Cache Ihres Browsers\n3. Versuchen Sie den Incognito-Modus\n4. Nutzen Sie 'Passwort vergessen' um ein neues zu setzen\n\nWenn Sie sich danach immer noch nicht einloggen koennen, sperren wir Ihr Konto voruebergehend und entsperren es sicher fuer Sie. Bitte melden Sie sich in diesem Fall mit Ihrer registrierten E-Mail-Adresse bei uns."
    },
    # --- Versandverzoegerung ---
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Mein Paket sollte vor 3 Tagen ankommen, ist aber noch nicht da.",
        "output": "Das verstehe ich — das ist aergerlich! Lassen Sie mich das pruefen.\n\nBitte teilen Sie mir Ihre Bestellnummer mit. In der Zwischenzeit koennen Sie selbst nachschauen:\n1. Gehen Sie zu 'Meine Bestellungen'\n2. Klicken Sie auf die Sendungsverfolgungsnummer\n3. Dort sehen Sie den aktuellen Standort Ihres Pakets\n\nSollte das Paket als 'zugestellt' markiert sein, aber nicht angekommen sein, melden wir einen Verlust beim Versanddienstleister und klaeren das fuer Sie!"
    },
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Die Sendungsverfolgung hat sich seit 5 Tagen nicht aktualisiert.",
        "output": "Eine eingefrorne Sendungsverfolgung ist beunruhigend — ich verstehe Ihre Sorge!\n\nMeistens passiert das wenn:\n- Das Paket in einem Verteilzentrum steckt\n- Ein technisches Problem beim Versanddienstleister vorliegt\n\nBitte teilen Sie mir Ihre Sendungsnummer mit. Wir kontaktieren den Versanddienstleister direkt und klaeren den Verbleib Ihres Pakets. Falls es verloren gegangen sein sollte, senden wir Ihnen kostenlos Ersatz — versprochen!"
    },
    # --- Kontaktdaten ---
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Ich moechte meine E-Mail-Adresse aendern.",
        "output": "Das koennen Sie ganz einfach selbst aendern:\n\n1. Einloggen und zu 'Einstellungen' > 'Persoenliche Daten' gehen\n2. Klicken Sie auf 'E-Mail-Adresse aendern'\n3. Geben Sie Ihre neue E-Mail-Adresse ein\n4. Bestaetigen Sie die Aenderung ueber den Link, den wir an Ihre neue Adresse senden\n\nWichtig: Bis zur Bestaetigung bleibt Ihre alte E-Mail-Adresse aktiv. Falls Sie Probleme haben, helfe ich Ihnen gerne weiter!"
    },
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Ich bin umgezogen und moechte meine Lieferadresse aktualisieren.",
        "output": "Herzlichen Glueckwunsch zum neuen Zuhause! Die Adresse koennen Sie schnell aendern:\n\n1. Gehen Sie zu 'Einstellungen' > 'Adressen'\n2. Klicken Sie auf 'Adresse bearbeiten' oder 'Neue Adresse hinzufuegen'\n3. Speichern Sie die Aenderung\n\nBitte beachten Sie: Laufende Bestellungen werden noch an die alte Adresse geliefert. Fuer zukuenftige Bestellungen wird automatisch die neue Adresse verwendet. Soll ich eine laufende Bestellung auf die neue Adresse umleiten?"
    },
    # --- Newsletter ---
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Ich moechte keine Marketing-E-Mails mehr erhalten.",
        "output": "Das verstehen wir vollkommen! So melden Sie sich ab:\n\n**Option 1 (schnellste):**\nKlicken Sie auf 'Abmelden' am Ende jeder unserer E-Mails\n\n**Option 2:**\n1. Einloggen und zu 'Einstellungen' > 'Benachrichtigungen' gehen\n2. Marketing-E-Mails deaktivieren\n\nBitte beachten: Wichtige E-Mails wie Bestellbestaetigung oder Versandbenachrichtigung erhalten Sie weiterhin. Die Abmeldung kann bis zu 48 Stunden dauern."
    },
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Ich habe mich vom Newsletter abgemeldet, bekomme aber immer noch E-Mails.",
        "output": "Das tut mir leid — das sollte nicht passieren!\n\nBitte teilen Sie mir mit:\n1. Wann haben Sie sich abgemeldet?\n2. An welche E-Mail-Adresse kommen die Mails noch?\n\nManchmal haben Kunden mehrere Adressen bei uns registriert. Wir pruefen das in unserem System und stellen sicher, dass alle Ihre Adressen aus dem Verteiler entfernt werden. Die endgueltige Abmeldung bestaetigen wir Ihnen per E-Mail."
    },
    # --- Bewertungen ---
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Wie kann ich ein Produkt bewerten?",
        "output": "Wir freuen uns sehr ueber Ihr Feedback!\n\nSo geben Sie eine Bewertung ab:\n1. Gehen Sie zur Produktseite des gekauften Artikels\n2. Scrollen Sie nach unten zu 'Kundenbewertungen'\n3. Klicken Sie auf 'Bewertung schreiben'\n4. Vergeben Sie 1-5 Sterne und schreiben Sie optional einen Kommentar\n\nAlternativ senden wir Ihnen nach jeder Lieferung eine automatische Bewertungsanfrage per E-Mail. Ihre ehrliche Meinung hilft anderen Kaeufer bei ihrer Entscheidung — vielen Dank!"
    },
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Ich moechte meine abgegebene Bewertung aendern oder loeschen.",
        "output": "Kein Problem! So bearbeiten oder loeschen Sie Ihre Bewertung:\n\n1. Loggen Sie sich ein und gehen Sie zu 'Mein Konto' > 'Meine Bewertungen'\n2. Finden Sie die entsprechende Bewertung\n3. Klicken Sie auf 'Bearbeiten' oder 'Loeschen'\n\nBitte beachten: Bewertungen koennen nur innerhalb von 90 Tagen nach Abgabe geaendert werden. Falls die Bewertung aelter ist oder Sie technische Probleme haben, helfe ich Ihnen gerne manuell weiter!"
    },
    # --- Empfehlungsprogramm ---
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Wie funktioniert euer Empfehlungsprogramm?",
        "output": "Unser Empfehlungsprogramm ist ganz einfach und lohnt sich fuer beide Seiten!\n\nSo funktioniert es:\n1. Gehen Sie zu 'Mein Konto' > 'Freunde einladen'\n2. Teilen Sie Ihren persoenlichen Empfehlungslink\n3. Sobald ein Freund seinen ersten Kauf abschliesst, erhalten Sie 10€ Gutschrift\n4. Ihr Freund bekommt ebenfalls 5€ Willkommensrabatt\n\nEs gibt keine Begrenzung — Sie koennen beliebig viele Freunde einladen. Ihre gesammelten Gutschriften sehen Sie jederzeit in Ihrem Konto unter 'Meine Gutschriften'."
    },
    {
        "instruction": "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.",
        "input": "Ich habe einen Freund eingeladen aber meine Praemie noch nicht erhalten.",
        "output": "Das pruefen wir gerne fuer Sie!\n\nBitte beachten Sie: Die Praemie wird erst gutgeschrieben wenn:\n1. Ihr Freund den Empfehlungslink verwendet hat (nicht einfach direkt registriert)\n2. Der erste Kauf abgeschlossen und bezahlt wurde\n3. Die Rueckgabefrist (30 Tage) abgelaufen ist\n\nBitte teilen Sie uns die E-Mail-Adresse Ihres eingeladenen Freundes mit — wir pruefen in unserem System ob alles korrekt verknuepft wurde und gutschreiben die Praemie manuell falls noetig."
    },
]

print(f"Dataset: {len(training_data)} examples")

Dataset: 28 examples


## 5. Format Dataset

Uses the **Alpaca prompt template** — a standard format for instruction fine-tuning.<br>
Each example is structured as `Instruction / Input / Response` and terminated with the EOS token so the model learns where to stop generating.

In [6]:
import random
from datasets import Dataset

ALPACA_PROMPT = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def format_examples(examples, tokenizer):
    EOS = tokenizer.eos_token
    texts = []
    for instruction, input_text, output in zip(
        examples["instruction"], examples["input"], examples["output"]
    ):
        texts.append(ALPACA_PROMPT.format(instruction, input_text, output) + EOS)
    return {"text": texts}

dataset = Dataset.from_list(training_data)
dataset = dataset.map(lambda x: format_examples(x, tokenizer), batched=True)

# Print random entry from dataset as example
idx = random.randint(0, len(dataset) - 1)
sample = dataset[idx]["text"]

print(f"Sample (index {idx}):\n")
for section in sample.split("###"):
    if section.strip():
        print(f"###{section.strip()}")
        print()

Map:   0%|          | 0/28 [00:00<?, ? examples/s]

Sample (index 20):

###Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

###Instruction:
Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell.

###Input:
Ich moechte meine E-Mail-Adresse aendern.

###Response:
Das koennen Sie ganz einfach selbst aendern:

1. Einloggen und zu 'Einstellungen' > 'Persoenliche Daten' gehen
2. Klicken Sie auf 'E-Mail-Adresse aendern'
3. Geben Sie Ihre neue E-Mail-Adresse ein
4. Bestaetigen Sie die Aenderung ueber den Link, den wir an Ihre neue Adresse senden

Wichtig: Bis zur Bestaetigung bleibt Ihre alte E-Mail-Adresse aktiv. Falls Sie Probleme haben, helfe ich Ihnen gerne weiter!<|im_end|>



## 6. Fine-Tune

Key hyperparameters:
- `per_device_train_batch_size=2` + `gradient_accumulation_steps=4` → effective batch size of 8
- `max_steps=120` — sufficient for this dataset size; loss convergence verified empirically
- `optim="adamw_8bit"` — memory-efficient optimizer from bitsandbytes
- `report_to="none"` — disables W&B/HF logging

In [7]:
from trl import SFTTrainer, SFTConfig
from transformers import DataCollatorForSeq2Seq

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=120,
        learning_rate=2e-4,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer_stats = trainer.train()

print(f"\nSteps:         {trainer_stats.global_step}")
print(f"Average training loss: {trainer_stats.training_loss:.4f}")
print(f"Runtime:       {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"Samples/sec:   {trainer_stats.metrics['train_samples_per_second']:.2f}")

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/28 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 28 | Num Epochs = 30 | Total steps = 120
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
10,2.269907
20,1.207576
30,0.733642
40,0.381516
50,0.125468
60,0.041299
70,0.026784
80,0.023310
90,0.021434
100,0.020463


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-120/tokenizer_config.json.



Steps:         120
Average training loss: 0.4076
Runtime:       332s
Samples/sec:   2.89


## 7. Qualitative Evaluation

Sanity check on held-out queries not seen during training.<br>
Systematic Teacher vs. Student benchmarking is in Notebook 04.

In [8]:
import transformers
transformers.logging.set_verbosity_error()
import warnings
warnings.filterwarnings("ignore")

FastLanguageModel.for_inference(model)

SYSTEM_INSTRUCTION = "Du bist ein hilfreicher deutscher Kundenservice-Assistent. Beantworte die Kundenanfrage hoeflich und professionell."

test_queries = [
    "Mein Paket ist seit 2 Wochen nicht angekommen.",
    "Kann ich auch per Rechnung bezahlen?",
    "Ich moechte meine Adresse aendern.",
    "Das Produkt entspricht nicht der Beschreibung.",
    "Gibt es das auch in Groesse XL?",
]

for query in test_queries:
    prompt = ALPACA_PROMPT.format(SYSTEM_INSTRUCTION, query, "")
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("=" * 60)
    print(f"Query:    {query}")
    print("=" * 60)
    print(f"Response: {response.split('### Response:')[-1].strip()}")

Query:    Mein Paket ist seit 2 Wochen nicht angekommen.
Response: Das verstehe ich — das ist aergerlich! Lassen Sie mich das pruefen.

Bitte teilen Sie mir Ihre Bestellnummer mit. In der Zwischenzeit koennen Sie selbst nachschauen:
1. Gehen Sie zu 'Meine Bestellungen'
2. Klicken Sie auf die Sendungsverfolgungsnummer
3. Dort sehen Sie den aktuellen Standort Ihres Pakets

Sollte das Paket als 'zugestellt' markiert sein, aber nicht angekommen sein, melden wir einen Verlust beim Versanddienstleister und klaeren das fuer Sie!
Query:    Kann ich auch per Rechnung bezahlen?
Response: Vielen Dank fuer Ihre Anfrage! Payment durch Rechnung ist bei uns aktiv. Sie koennen das so organisieren:

1. Eingabefelder 'Zahlungsart' leeren
2. 'Rechnungsende' aktivieren
3. Die送货到地址 senden

Bitte beachten: Die Rechnung wird um 14个工作天送达，并需自费。如有其他问题，敬请告知！
Query:    Ich moechte meine Adresse aendern.
Response: Das koennen Sie ganz einfach selbst aendern:

1. Einloggen und zu 'Einstellungen' > 'Adressen' gehen


## 8. Push to Hugging Face Hub

Pushes the LoRA adapter and tokenizer to Hugging Face. Switch `if False` to `if True` to activate a section.

**Kaggle:** add `HF_TOKEN` via *Add-ons → Secrets*.  
**Local:** set `HF_TOKEN` as an environment variable.

In [9]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads .env locally, no-op on Kaggle

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

HF_TOKEN = os.getenv("HF_TOKEN")
REPO_ID = "Feyerade/german-support-qwen-lora-adapter"

# Save adapter to disk
# On Kaggle: writes to /kaggle/working/ (ephemeral — download manually or push to Hub)
# Local:     writes to the current working directory
if False:
    # Save trained adapter only
    model.save_pretrained_merged(
        save_directory = "german_support_lora",
        tokenizer = tokenizer,
        save_method = "lora",
    )
    print("LoRA Adapter saved locally!")

if False:
    silver_model.save_pretrained_merged(
        save_directory = "german_support_merged",
        tokenizer = tokenizer,
        save_method = "merged_16bit",
    )
    print("Merged Model saved!")

# Push to Hugging Face Hub
if False:
    model.push_to_hub(REPO_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(REPO_ID, token=HF_TOKEN)
    print(f"Pushed to: https://huggingface.co/{REPO_ID}")